## Text-to-SQL Enterprise Agent (MVP)

This notebook builds a local SQLite database (`university_agent.db`) with a dummy University dataset and implements a safe Text-to-SQL pipeline.

**Architectural constraints covered:**
- Local SQLite only
- No RAG: static schema injection (DDL extraction)
- Strict separation: LLM generates SQL text only; Python executes
- Safety first: deterministic rule-based blocking of data-modifying SQL

**Before you run**
- Install dependencies: `pip install -r requirements.txt` (includes `pandas`, `google-generativeai`, `python-dotenv`)
- Put your Gemini API key in the project `.env` file: `GEMINI_API_KEY=your_key_here` (file is gitignored). Cell 1 loads it via `load_dotenv()`.


In [ ]:
# Imports + load `.env` (GEMINI_API_KEY)

from __future__ import annotations

import os
import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai

# Load secrets from `.env` in the project folder.
load_dotenv(Path.cwd() / ".env")


True

In [ ]:
# Dummy dataset + SQLite database creation
# Optional: create a dummy dataset

def create_dummy_university_data(seed: int = 7) -> dict[str, pd.DataFrame]:
    """Create small, relational University tables.

    Trade-offs / report notes:
    - Synthetic data is ideal for demos and reproducibility.
    - The schema is intentionally small; static schema injection becomes costly
      as schemas grow (prompt length, latency, token cost).
    """
    rng = pd.Series(range(1, 21))  # 20 students

    students = pd.DataFrame(
        {
            "student_id": rng,
            "full_name": [f"Student {i:02d}" for i in rng],
            "email": [f"student{i:02d}@example.edu" for i in rng],
            "enrollment_year": [2022 + (i % 4) for i in rng],
            "major": [["CS", "DS", "IT", "Business", "Math"][i % 5] for i in rng],
        }
    )

    courses = pd.DataFrame(
        {
            "course_id": [101, 102, 103, 201, 202, 301],
            "course_code": ["CS101", "DS102", "IT103", "CS201", "BUS202", "MATH301"],
            "course_name": [
                "Intro to Programming",
                "Data Fundamentals",
                "Networks Basics",
                "Databases",
                "Business Analytics",
                "Linear Algebra",
            ],
            "credits": [6, 6, 6, 6, 6, 6],
        }
    )

    # Many-to-many between students and courses.
    # Each student takes 3 courses.
    enroll_rows: list[dict[str, Any]] = []
    grade_scale = ["HD", "D", "C", "P", "F"]

    for student_id in students["student_id"].tolist():
        course_choices = [
            courses.loc[(student_id + k) % len(courses), "course_id"]
            for k in (0, 2, 4)
        ]
        for course_id in course_choices:
            grade = grade_scale[(student_id + int(course_id)) % len(grade_scale)]
            score = int(55 + ((student_id * 7 + int(course_id)) % 46))  # 55..100
            enroll_rows.append(
                {
                    "student_id": int(student_id),
                    "course_id": int(course_id),
                    "semester": ["2024S1", "2024S2"][student_id % 2],
                    "grade": grade,
                    "score": score,
                }
            )

    grades = (
        pd.DataFrame(enroll_rows)
        .sort_values(["student_id", "course_id"])
        .reset_index(drop=True)
    )

    return {"students": students, "courses": courses, "grades": grades}


def write_university_db(db_path: str = "university_agent.db") -> str:
    """Create (or overwrite) a local SQLite DB and populate it.

    Trade-offs / report notes:
    - For MVP speed, we use pandas `to_sql`.
    - We add PK/FK constraints after writing using a rename/recreate pattern
      (SQLite has limited ALTER TABLE support).
    """
    data = create_dummy_university_data()

    if os.path.exists(db_path):
        os.remove(db_path)

    with sqlite3.connect(db_path) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")

        data["students"].to_sql("students", conn, index=False)
        data["courses"].to_sql("courses", conn, index=False)
        data["grades"].to_sql("grades", conn, index=False)

        conn.executescript(
            """
            PRAGMA foreign_keys = OFF;

            ALTER TABLE students RENAME TO students_old;
            CREATE TABLE students (
                student_id INTEGER PRIMARY KEY,
                full_name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                enrollment_year INTEGER NOT NULL,
                major TEXT NOT NULL
            );
            INSERT INTO students SELECT * FROM students_old;
            DROP TABLE students_old;

            ALTER TABLE courses RENAME TO courses_old;
            CREATE TABLE courses (
                course_id INTEGER PRIMARY KEY,
                course_code TEXT NOT NULL UNIQUE,
                course_name TEXT NOT NULL,
                credits INTEGER NOT NULL
            );
            INSERT INTO courses SELECT * FROM courses_old;
            DROP TABLE courses_old;

            ALTER TABLE grades RENAME TO grades_old;
            CREATE TABLE grades (
                student_id INTEGER NOT NULL,
                course_id INTEGER NOT NULL,
                semester TEXT NOT NULL,
                grade TEXT NOT NULL,
                score INTEGER NOT NULL,
                PRIMARY KEY (student_id, course_id, semester),
                FOREIGN KEY (student_id) REFERENCES students(student_id),
                FOREIGN KEY (course_id) REFERENCES courses(course_id)
            );
            INSERT INTO grades SELECT * FROM grades_old;
            DROP TABLE grades_old;

            PRAGMA foreign_keys = ON;
            """
        )

    return db_path


# Create the DB now
db_path = write_university_db("university_agent.db")
print("DB created at:", db_path)


DB created at: university_agent.db


In [ ]:
# Function: Schema extraction (Static Schema Injection)

def get_schema(db_path: str) -> str:
    """Extract CREATE TABLE statements for all user tables in SQLite.

    Trade-offs / report notes:
    - Static injection is simple and deterministic (no retrieval system).
    - It scales poorly: large schemas increase prompt length, latency, and cost.
    - For a 1-week MVP and small schemas, it is an excellent baseline.
    """
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            """
            SELECT sql
            FROM sqlite_master
            WHERE type='table'
              AND name NOT LIKE 'sqlite_%'
              AND sql IS NOT NULL
            ORDER BY name;
            """
        ).fetchall()

    ddl_statements = [r[0].strip().rstrip(";") + ";" for r in rows]
    return "\n\n".join(ddl_statements)


schema_text = get_schema(db_path)
print(schema_text)


CREATE TABLE courses (
                course_id INTEGER PRIMARY KEY,
                course_code TEXT NOT NULL UNIQUE,
                course_name TEXT NOT NULL,
                credits INTEGER NOT NULL
            );

CREATE TABLE grades (
                student_id INTEGER NOT NULL,
                course_id INTEGER NOT NULL,
                semester TEXT NOT NULL,
                grade TEXT NOT NULL,
                score INTEGER NOT NULL,
                PRIMARY KEY (student_id, course_id, semester),
                FOREIGN KEY (student_id) REFERENCES students(student_id),
                FOREIGN KEY (course_id) REFERENCES courses(course_id)
            );

CREATE TABLE students (
                student_id INTEGER PRIMARY KEY,
                full_name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                enrollment_year INTEGER NOT NULL,
                major TEXT NOT NULL
            );


In [ ]:
# Function: generate_sql - LLM SQL generation (Gemini)

SQL_TRANSLATION_SYSTEM_PROMPT = """\
You are an expert data analyst and SQL translator.
Your ONLY job is to translate the user's question into a SINGLE SQLite SELECT query.

Rules (must follow):
- Output ONLY the SQL query text. No markdown fences, no explanations.
- Use ONLY the tables and columns that exist in the provided schema.
- Generate READ-ONLY SQL: SELECT queries only.
- Do NOT use any data-modifying statements: INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, REPLACE, TRUNCATE, VACUUM, PRAGMA, ATTACH, DETACH.
- Do NOT reference sqlite_master or any internal SQLite tables.
- Prefer simple SQL compatible with SQLite.

If the question cannot be answered using the schema, output exactly:
SELECT 'UNANSWERABLE_WITH_GIVEN_SCHEMA' AS error;
"""


def generate_sql(user_question: str, schema_text: str, *, model_name: str = "gemini-1.5-flash") -> str:
    """Call Gemini to generate SQL from a question + schema.

    Strict separation guarantee:
    - The LLM sees ONLY schema DDL (no row data).
    - The LLM returns ONLY SQL text; Python decides if/when to execute.

    Trade-offs / report notes:
    - LLM output can be invalid SQL or reference non-existent columns.
      We handle this with execution-time error handling and safety checks.
    - Prompt-injection risk exists (e.g., user asks to DROP TABLE). We still
      deterministically block dangerous tokens before execution.
    """

    # Populated from `.env` via Cell 1 (`load_dotenv`). Never commit real keys.
    api_key = (os.environ.get("GEMINI_API_KEY") or "").strip()
    if not api_key:
        raise ValueError(
            "GEMINI_API_KEY is missing. Set GEMINI_API_KEY in `.env` and rerun Cell 1."
        )
    genai.configure(api_key=api_key)

    model = genai.GenerativeModel(
        model_name,
        system_instruction=SQL_TRANSLATION_SYSTEM_PROMPT,
    )

    prompt = f"""\
### SQLite schema (DDL)
{schema_text}

### User question
{user_question}
"""

    try:
        response = model.generate_content(
            contents=prompt,
            generation_config={
                "temperature": 0.0,
                "max_output_tokens": 512,
            },
        )

        sql = (response.text or "").strip()

        sql = re.sub(r"^```(?:sql)?\s*", "", sql, flags=re.IGNORECASE).strip()
        sql = re.sub(r"\s*```$", "", sql).strip()

        return sql
    except Exception as e:
        return f"Error connecting to Gemini API: {e}"


In [ ]:
# Function: CSV ingestion utility (CSV -> SQLite)

def ingest_csvs_to_db(csv_file_paths: list[str], output_db_path: str = "dynamic_agent.db") -> str:
    """Ingest one or more CSV files into a local SQLite database.

    Requirements implemented:
    - Iterate through `csv_file_paths`
    - For each CSV: pandas -> DataFrame -> SQLite table
    - Table name = CSV base name without extension (financials.csv -> financials)
    - Replace table if it already exists
    - Return the output DB path

    Trade-offs / report notes:
    - `to_sql(if_exists='replace')` is convenient but will not preserve indexes,
      constraints, or column types perfectly. For an MVP this is acceptable.
    - For production, you'd add explicit DDL (types, keys) and validation.
    """
    if not csv_file_paths:
        raise ValueError("csv_file_paths must contain at least one CSV path")

    out_path = Path(output_db_path)
    if out_path.exists():
        out_path.unlink()  # create a fresh DB each run

    with sqlite3.connect(str(out_path)) as conn:
        for csv_path_str in csv_file_paths:
            csv_path = Path(csv_path_str)
            if not csv_path.exists():
                raise FileNotFoundError(f"CSV not found: {csv_path}")

            table_name = csv_path.stem  # base name without .csv
            df = pd.read_csv(csv_path)

            # Replace table if it exists
            df.to_sql(table_name, conn, if_exists="replace", index=False)

    return str(out_path)


In [ ]:
# Function: Deterministic safety gate (block dangerous SQL)

_DANGEROUS_SQL_PATTERN = re.compile(
    r"""
    (?ix)                       # i=case-insensitive, x=verbose
    \b(
        insert|update|delete|drop|alter|create|replace|truncate|
        vacuum|pragma|attach|detach|reindex|analyze|
        begin|commit|rollback|savepoint|release
    )\b
    """.strip()
)


def is_safe_query(sql_string: str) -> bool:
    """Deterministic rule-based safety filter.

    Security stance:
    - Fail closed. Allow only read-only queries.
    - Block anything that looks like write/DDL/transaction SQL.

    Trade-offs / report notes:
    - This conservative keyword policy may create false positives.
      For an MVP, that's acceptable because safety > completeness.
    - A SQL parser can help, but still requires careful policy design.
    """
    if not sql_string or not sql_string.strip():
        return False

    s = sql_string.strip().rstrip(";").strip()

    # Must start with SELECT or WITH (CTE) for a read-only query.
    if not re.match(r"(?is)^(select|with)\b", s):
        return False

    # Block dangerous tokens anywhere.
    if _DANGEROUS_SQL_PATTERN.search(s) is not None:
        return False

    # Block internal SQLite metadata tables.
    if re.search(r"(?is)\bsqlite_master\b|\bsqlite_schema\b", s):
        return False

    return True


In [ ]:
# Function:SQLite execution (Python runs SQL, not the LLM)

@dataclass(frozen=True)
class QueryResult:
    columns: list[str]
    rows: list[tuple[Any, ...]]


def execute_query(db_path: str, sql_string: str) -> QueryResult:
    """Execute a SQL query and fetch all rows.

    Trade-offs / report notes:
    - `fetchall()` is fine for MVP and small datasets.
    - For large results, you would paginate or stream.
    """
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        cursor = conn.execute(sql_string)
        rows = cursor.fetchall()
        columns = [d[0] for d in cursor.description] if cursor.description else []
        return QueryResult(columns=columns, rows=[tuple(r) for r in rows])


In [ ]:
# # Cell 7: Step 3 — Master wrapper function

# def ask_database(
#     question: str,
#     *,
#     db_path: str = "university_agent.db",
#     model_name: str = "gemini-2.5-flash",
# ) -> QueryResult:
#     """End-to-end agent wrapper:
#     schema extraction -> LLM SQL generation -> safety validation -> execution.

#     Error handling principles:
#     - Fail closed: unsafe or invalid SQL is not executed.
#     - Notebook-friendly errors: return a 1-row result with an error message.
#       (Alternative design: raise exceptions and let a UI handle them.)
#     """
#     if not os.path.exists(db_path):
#         # Notebook convenience: auto-create if missing.
#         write_university_db(db_path)

#     try:
#         schema_text = get_schema(db_path)
#         sql = generate_sql(question, schema_text, model_name=model_name)

#         if "UNANSWERABLE_WITH_GIVEN_SCHEMA" in sql:
#             return QueryResult(columns=["error"], rows=[("UNANSWERABLE_WITH_GIVEN_SCHEMA",)])

#         if not is_safe_query(sql):
#             return QueryResult(columns=["error", "sql"], rows=[("BLOCKED_UNSAFE_SQL", sql)])

#         return execute_query(db_path, sql)

#     except Exception as e:
#         return QueryResult(columns=["error"], rows=[(f"{type(e).__name__}: {e}",)])


In [ ]:
# Function: Refactor ask_database to require an existing DB

def ask_database(
    question: str,
    *,
    db_path: str = "university_agent.db",
    model_name: str = "gemini-2.5-flash",
) -> QueryResult:
    """End-to-end agent wrapper (DB must already exist).

    Changes vs previous version:
    - No auto-creation of a synthetic university DB.
    - If `db_path` does not exist, raise FileNotFoundError.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError("input database not found")

    try:
        schema_text = get_schema(db_path)
        sql = generate_sql(question, schema_text, model_name=model_name)

        if "UNANSWERABLE_WITH_GIVEN_SCHEMA" in sql:
            return QueryResult(columns=["error"], rows=[("UNANSWERABLE_WITH_GIVEN_SCHEMA",)])

        if not is_safe_query(sql):
            return QueryResult(columns=["error", "sql"], rows=[("BLOCKED_UNSAFE_SQL", sql)])

        return execute_query(db_path, sql)

    except Exception as e:
        return QueryResult(columns=["error"], rows=[(f"{type(e).__name__}: {e}",)])


In [ ]:
#Test

# --- Workflow A: Ask an existing .db file (e.g., the university_agent.db created earlier)
print("Workflow A: existing DB")
res_a = ask_database(
    "How many students are enrolled in each major?",
    db_path="university_agent.db",
)
print(res_a.columns)
print(res_a.rows[:10])


# --- Workflow B: Ingest user-provided CSVs into a new .db, then ask it
print("\nWorkflow B: CSV -> DB -> ask_database")

# Create two dummy CSVs (stand-in for user uploads)
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

customers_csv = data_dir / "customers.csv"
sales_csv = data_dir / "sales.csv"

pd.DataFrame(
    {
        "customer_id": [1, 2, 3],
        "name": ["Alice", "Bob", "Charlie"],
        "segment": ["SMB", "Enterprise", "SMB"],
    }
).to_csv(customers_csv, index=False)

pd.DataFrame(
    {
        "sale_id": [1001, 1002, 1003, 1004],
        "customer_id": [1, 2, 2, 3],
        "amount": [120.0, 550.0, 75.0, 220.0],
        "region": ["APAC", "APAC", "EU", "APAC"],
    }
).to_csv(sales_csv, index=False)

# Ingest into SQLite (tables: customers, sales)
dynamic_db = ingest_csvs_to_db([str(customers_csv), str(sales_csv)], output_db_path="dynamic_agent.db")
print("Created:", dynamic_db)

# Ask questions against the dynamic DB
res_b = ask_database(
    "What is the total sales amount by region?",
    db_path=dynamic_db,
)
print(res_b.columns)
print(res_b.rows[:10])


Workflow A: existing DB
['major', 'COUNT(student_id)']
[('Business', 4), ('CS', 4), ('DS', 4), ('IT', 4), ('Math', 4)]

Workflow B: CSV -> DB -> ask_database
Created: dynamic_agent.db
['region', 'SUM(amount)']
[('APAC', 890.0), ('EU', 75.0)]


In [ ]:
# Function: Auto-workflow router (DB vs CSV -> DB)

def ask_from_files(
    question: str,
    file_paths: list[str] | str,
    *,
    output_db_path,
    model_name: str,
) -> QueryResult:
    """Automatically choose the correct workflow based on file extension.

    Behavior:
    - If `file_paths` is a single `.db` file: query it directly with `ask_database()`.
    - If `file_paths` is one or more `.csv` files: ingest them into `output_db_path`, then query.

    Notes:
    - Mixed inputs (both .db and .csv) are rejected to avoid ambiguity.
    - This keeps the core `ask_database(db_path=...)` contract simple.
    """
    if isinstance(file_paths, str):
        paths = [file_paths]
    else:
        paths = list(file_paths)

    if not paths:
        raise ValueError("file_paths must contain at least one path")

    exts = {Path(p).suffix.lower() for p in paths}

    if exts == {".db"}:
        if len(paths) != 1:
            raise ValueError("Provide exactly one .db file path")
        return ask_database(question, db_path=paths[0], model_name=model_name)

    if exts == {".csv"}:
        db_path = ingest_csvs_to_db(paths, output_db_path=output_db_path)
        return ask_database(question, db_path=db_path, model_name="gemini-2.5-flash")

    raise ValueError(f"Unsupported or mixed file types: {sorted(exts)}")

In [ ]:
#Test 2
res1 = ask_from_files("How many students are enrolled in each major?", "data/university_agent.db")
res2 = ask_from_files("What is total amount by region?", ["data/customers.csv", "data/sales.csv"])
print(res1.columns, res1.rows[:5])
print(res2.columns, res2.rows[:5])

['major', 'COUNT(student_id)'] [('Business', 4), ('CS', 4), ('DS', 4), ('IT', 4), ('Math', 4)]
['region', 'SUM(amount)'] [('APAC', 890.0), ('EU', 75.0)]


In [81]:
print(res1)

QueryResult(columns=['major', 'COUNT(student_id)'], rows=[('Business', 4), ('CS', 4), ('DS', 4), ('IT', 4), ('Math', 4)])


In [ ]:
# # Cell 8: Quick tests
# # Requires `GEMINI_API_KEY` in `.env` (loaded in Cell 1). Missing key raises ValueError from generate_sql.

# questions = [
#     "How many students are enrolled in each major?",
#     "List the top 5 students by average score.",
#     "Show each course and the number of distinct students who took it.",
#     "DROP TABLE students;",  # should be blocked by safety gate
# ]

# for q in questions:
#     print("\nQ:", q)
#     result = ask_database(q, db_path=db_path)
#     print("Columns:", result.columns)
#     print("Rows (first 10):", result.rows[:10])



Q: How many students are enrolled in each major?
Columns: ['major', 'COUNT(student_id)']
Rows (first 10): [('Business', 4), ('CS', 4), ('DS', 4), ('IT', 4), ('Math', 4)]

Q: List the top 5 students by average score.
Columns: ['full_name', 'average_score']
Rows (first 10): [('Student 10', 91.66666666666667), ('Student 09', 89.33333333333333), ('Student 16', 87.66666666666667), ('Student 18', 86.33333333333333), ('Student 15', 85.33333333333333)]

Q: Show each course and the number of distinct students who took it.
Columns: ['course_name', 'COUNT(DISTINCT T2.student_id)']
Rows (first 10): [('Intro to Programming', 10), ('Data Fundamentals', 10), ('Networks Basics', 10), ('Databases', 10), ('Business Analytics', 10), ('Linear Algebra', 10)]

Q: DROP TABLE students;
Columns: ['error']
Rows (first 10): [('UNANSWERABLE_WITH_GIVEN_SCHEMA',)]


In [ ]:
# questions = [
#     "Explain what is reinforcement learning",
#     "Summarise what the university_agent dataset is about?",
# ]

# for q in questions:
#     print("\nQ:", q)
#     result = ask_database(q, db_path=db_path)
#     print("Columns:", result.columns)
#     print("Rows (first 10):", result.rows[:10])


Q: Explain what is reinforcement learning
Columns: ['error']
Rows (first 10): [('UNANSWERABLE_WITH_GIVEN_SCHEMA',)]

Q: Summarise what the university_agent dataset is about?
Columns: ['error']
Rows (first 10): [('UNANSWERABLE_WITH_GIVEN_SCHEMA',)]
